# 👋 Welcome
This notebook is here to give you a hands-on intro to my project and its code. If you have not read the readme please do so and come back here.
I have implemented the WkNN algorithm for position estimation based on 5G RF (radio frequency) signals.
At the end of this project I did not have a lot of time to clean up the code since I was writing the thesis, maybe you want to do some cleanup work?
But I did create a 'wrapper' to interface with the model that I will use below here.

I will show some key things to get started with running the code and making your own experiments and changes.

**This notebook covers**
1. Loading the dataset
2. Filtering dataset
3. Filtering features
4. Running the model/experiment
5. Storing the results

Have fun!

## Load the dataset
I was origianlly working with a NB-IoT dataset, and extended the data loader to handle both datasets. That is why you need to pass a NETWORK_TYPE. This could probably be separate functions, but oh well.

In [ ]:
import pandas as pd

from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, RF_PARAM_5G

filename = "5G_data_2023.mat"
df = load_dataframe(filename, NETWORK_TYPE._5G)

df.info()

## Filtering the Dataframe
The following code removes unused data to save space, in addition to filtering data based on experiment configuration.

In [ ]:
from scripts.beamforming import matrix_filter
from scripts.data_filter import filter_dataframe
from scripts.utils import get_arfcns_from_bands

## Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

config = {
    'rf_param': RF_PARAM_5G.RSRQ,  # use the RSRQ RF feature
    'frequency_bands': [78],  # only use the n78 band
    'operators': [1, 10, 50, 88],  # use all the operators
    'campaigns': list(range(0, 21)),  # Use campaigns 0 - 20
    'n_best_pcis': None,  # Use all PCIs
    'n_best_beams': 4  # Use best 4 beams
}

include_cols = [
    "pci",
    "beam_index",
    "nr_arfcn",
    "operator_id",
    config['rf_param'].value
]

# Filters data from the measuremet matrices based on the config params above
df = filter_dataframe(
    df=df,
    operators=config['operators'],
    include_columns=include_cols,
    freqs=get_arfcns_from_bands(config['frequency_bands']),
    campaigns=config['campaigns'],

)

# Experiment-specific, you can filter the dataset to run the algorithm using
# specific beam-pci configurations.
# n_best_beams or n_best_pcis value of None, means use all (or don't filter).

if config['n_best_pcis'] or config['n_best_beams']:
    df.loc[:, "measurements_matrix"] = df.loc[:, "measurements_matrix"].apply(
        lambda x: matrix_filter(
            x,
            config['rf_param'],
            include_n_best_pcis=config['n_best_pcis'],
            include_n_best_beams=config['n_best_beams'],
        )
    )

df.info()

## Run Experiment
This example show how you can run the model with default settings. I'm using the random seeds for reproducibility. I also have some examples in other experiments using multiple threads for more efficient testing, however this can be challenging due to memory usage, and can also lead to less reliabele runtime measurements.

In [ ]:
from scripts.localization_model import LocalizationModel
from scripts.utils import dataset_tp_rp_split
import numpy as np

data = []
n_runs = 5

random_seeds = np.loadtxt('../data/random_seeds.csv', dtype=int)

for i in range(n_runs):
    print(f'Run #{i + 1} / {n_runs}')
    # Split the dataset (70% RPs, 30% TPs)
    df_tp, df_rp = dataset_tp_rp_split(df, 0.3, random_seeds[i])

    # Init the model and train based on RPs
    loc_model = LocalizationModel()
    loc_model.fit(df_rp)

    # Predict positions of TPs
    est_positions = loc_model.predict(df_tp)

    # Calculate errors and get metrics (stats)
    error, stats = loc_model.get_performance_stats(df_tp, est_positions, print_stats=False)

    data.append(stats)

results = pd.DataFrame(data)


## Storing the results
This is how you can print the results to get an overview, and store all the
results. The **save_experiment_result** will store the config as a .json file, and the dataframe(s) as .csv files. You can put multiple dataframes into the results dict to store more. The results will be stored in **data/results/experiemnts/{title}**. Duplicate experiment names will be appended with a number like **get_started_experiment_1**

In [ ]:
from scripts.data_writer import save_experiment_result

print(f"""
==== EXPERIMENT SUMMARY ====

[CONFIG]
{config}

[RESULTS]
{results.mean().round(2)}
""")

# It can be useful to add more metadata in the config, so you remember what the results were when you go to plot them.

config['n_runs'] = n_runs
config['model_settings'] = 'default'

save_experiment_result(
    title="get_started_experiment",
    config=config,
    results={
        'results': results,
    },
)

## 👏 That's it!
That should cover the basics needed to get started.
Good luck!
